# Task 3: Database Storage and SQL Analytics

## Overview

This notebook loads the enriched dataset from Task 2 into a structured relational database, then runs analytical SQL queries that produce insights for the final report.

### Why DuckDB instead of Oracle XE

| Criterion | Oracle XE | DuckDB |
|---|---|---|
| Kaggle setup | Requires system install, often fails | Single `pip install`, zero config |
| Query language | Standard SQL | Standard SQL + window functions |
| Analytical performance | OLTP-optimized, slow on aggregations | Columnar OLAP engine, fast on aggregations |
| Python integration | `oracledb` driver, connection string | `duckdb.connect()`, reads pandas natively |
| Portability | Requires Oracle installation everywhere | Single `.duckdb` file, runs anywhere |

### Primary goals
- Design and create a normalized relational schema (3 tables)
- Load all 2,282 reviews with full NLP enrichment
- Write 6 analytical SQL queries using window functions
- Export query results as CSVs for Task 4 visualizations
- Export a SQL dump for GitHub submission

### Success metrics

| Metric | Target |
|---|---|
| Rows inserted | 2,282 (all reviews) |
| Schema validation | All foreign keys resolve |
| SQL dump committed | `.sql` file exported |
| Window function queries | 5 minimum |
| Query result CSVs | 1 per analytical query |

---
## Step 1 — Install and Import

### Library roles

```
duckdb        → in-process analytical database engine
pandas        → data manipulation and CSV I/O
numpy         → numeric operations and NaN handling
pathlib       → safe cross-platform file path handling
```

> **Kaggle note:** DuckDB is not pre-installed on Kaggle. The install takes under 10 seconds.

In [31]:
!pip install duckdb -q

import os
import warnings
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import duckdb

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print(f"DuckDB version   : {duckdb.__version__}")
print(f"pandas version   : {pd.__version__}")
print(f"Ready            : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}") 

DuckDB version   : 1.3.2
pandas version   : 2.3.3
Ready            : 2026-03-22 04:56:04


---
## Step 2 — Configuration

### File layout

```
Input (from Task 2)
  reviews_enriched.csv     full NLP-enriched dataset
  keywords_per_bank.csv    TF-IDF keywords per bank
  topics_summary.csv       BERTopic topic metadata

Output
  bank_reviews.duckdb      the database file (submit with GitHub)
  bank_reviews_dump.sql    SQL schema + inserts (commit to repo)
  query_results/           one CSV per analytical query (for Task 4)
```

In [32]:
DATA_DIR   = Path("/kaggle/working/b5w2_outputs")
OUTPUT_DIR = Path("/kaggle/working/b5w2_outputs")
QUERY_DIR  = OUTPUT_DIR / "query_results"
QUERY_DIR.mkdir(parents=True, exist_ok=True)

ENRICHED_CSV  = DATA_DIR / "reviews_enriched.csv"
KEYWORDS_CSV  = DATA_DIR / "keywords_per_bank.csv"
TOPICS_CSV    = DATA_DIR / "topics_summary.csv"
DB_PATH       = OUTPUT_DIR / "bank_reviews.duckdb"
SQL_DUMP_PATH = OUTPUT_DIR / "bank_reviews_dump.sql"

if DB_PATH.exists():
    DB_PATH.unlink()
    print("Removed existing database for clean rebuild.")

print(f"Database path    : {DB_PATH}")
print(f"SQL dump path    : {SQL_DUMP_PATH}")
print(f"Query output dir : {QUERY_DIR}")

Database path    : /kaggle/working/b5w2_outputs/bank_reviews.duckdb
SQL dump path    : /kaggle/working/b5w2_outputs/bank_reviews_dump.sql
Query output dir : /kaggle/working/b5w2_outputs/query_results


---
## Step 3 — Load and Fix Enriched Data

### Two fixes applied before inserting

**Fix 1 — Unnamed topics:** BERTopic's second run produced 14 topics but `TOPIC_NAMES` only covered 11. Two topics kept raw keyword labels. We map them here so `topic_name` is clean throughout the database.

**Fix 2 — `low_confidence` dtype:** When pandas writes a boolean to CSV and reads it back, it returns the string `"True"/"False"`. SQL engines treat these differently from boolean `TRUE/FALSE`. We cast before insert.

**Fix 3 — `topic_id` as integer:** DuckDB infers `DOUBLE` from CSVs with `NaN` values. We fill `-1` (the outlier sentinel) and cast to `INTEGER`.

### Why `-1` as outlier sentinel instead of NULL

`NULL` in SQL means unknown. Topic `-1` is explicitly the outlier class BERTopic assigned. Using `-1` keeps it visible in aggregations and makes `WHERE topic_id != -1` a clean filter.

In [33]:
df = pd.read_csv(ENRICHED_CSV)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print(f"Loaded: {len(df):,} rows, {df.shape[1]} columns")

# Fix 1: unnamed topics
UNNAMED_TOPIC_MAP = {
    "boa | app | this | the":       "BOA General Complaints",
    "amole | otp | it | is":        "Amole and OTP Issues",
    "አይሰራም | በጣም | ነው | ለምንድነው":   "Amharic Complaints",
    "ነው | በጣም | ግን | ብር":          "Amharic Reviews",
}
df["topic_name"] = df["topic_name"].replace(UNNAMED_TOPIC_MAP)
unnamed = df["topic_name"].str.contains(r"\|", na=False).sum()
print(f"Unnamed topics remaining: {unnamed}")

# Fix 2: low_confidence dtype
df["low_confidence"] = (
    df["low_confidence"].astype(str).str.lower()
    .map({"true": True, "false": False, "nan": False})
    .fillna(False).astype(bool)
)

# Fix 3: topic_id as integer
df["topic_id"] = df["topic_id"].fillna(-1).astype(int)

# Fix 4: other boolean columns
for col in ["is_short_review", "developer_replied"]:
    df[col] = (
        df[col].astype(str).str.lower()
        .map({"true": True, "false": False})
        .fillna(False).astype(bool)
    )

# Fix 5: fill NaN strings
for col in ["sentiment_label","topic_name","language","review_lemma","developer_reply","user_name"]:
    if col in df.columns:
        df[col] = df[col].fillna("")

print("\nDtypes after fixes:")
print(df[["low_confidence","is_short_review","developer_replied","topic_id"]].dtypes.to_string())
print("\nTopic name distribution (top 8):")
print(df["topic_name"].value_counts().head(8).to_string())

Loaded: 2,282 rows, 20 columns
Unnamed topics remaining: 86

Dtypes after fixes:
low_confidence        bool
is_short_review       bool
developer_replied     bool
topic_id             int64

Topic name distribution (top 8):
topic_name
                              548
Uncategorized                 316
General App Complaints        275
General Banking Experience    267
Dashen Super App Feedback     208
Speed and Convenience         206
Positive General Feedback     133
Amharic Reviews                54


---
## Step 4 — Database Schema Design

### Three-table normalized schema

```
banks                          topics
bank_id    PK  INTEGER         topic_id    PK  INTEGER
bank_name      VARCHAR         topic_name      VARCHAR
app_id         VARCHAR         topic_size      INTEGER
known_rating   DOUBLE          keywords        VARCHAR
      |                               |
      +------------ reviews ----------+
                   review_id   PK  VARCHAR
                   bank_id     FK  → banks
                   topic_id    FK  → topics
                   review          VARCHAR
                   rating          INTEGER
                   date            DATE
                   helpful_count   INTEGER
                   language        VARCHAR
                   sentiment_label VARCHAR
                   sentiment_score DOUBLE
                   low_confidence  BOOLEAN
                   is_short_review BOOLEAN
                   developer_replied BOOLEAN
                   topic_name      VARCHAR  (denormalized convenience)
                   topic_prob      DOUBLE
                   source          VARCHAR
```

### Why `topic_name` is denormalized in reviews

Strictly normalized design requires a JOIN to `topics` for every query that needs a name. Since most analytical queries group by topic name, keeping it in `reviews` as a convenience column avoids repetitive JOINs. The FK `topic_id` still enforces referential integrity.

In [34]:
con = duckdb.connect(str(DB_PATH))

con.execute("""
    CREATE TABLE banks (
        bank_id      INTEGER PRIMARY KEY,
        bank_name    VARCHAR NOT NULL,
        app_id       VARCHAR NOT NULL,
        known_rating DOUBLE
    )
""")

con.execute("""
    CREATE TABLE topics (
        topic_id   INTEGER PRIMARY KEY,
        topic_name VARCHAR NOT NULL,
        topic_size INTEGER,
        keywords   VARCHAR
    )
""")

con.execute("""
    CREATE TABLE reviews (
        review_id         VARCHAR PRIMARY KEY,
        bank_id           INTEGER REFERENCES banks(bank_id),
        topic_id          INTEGER REFERENCES topics(topic_id),
        review            VARCHAR,
        rating            INTEGER CHECK (rating BETWEEN 1 AND 5),
        date              DATE,
        helpful_count     INTEGER DEFAULT 0,
        language          VARCHAR,
        sentiment_label   VARCHAR,
        sentiment_score   DOUBLE,
        low_confidence    BOOLEAN DEFAULT FALSE,
        is_short_review   BOOLEAN DEFAULT FALSE,
        developer_replied BOOLEAN DEFAULT FALSE,
        topic_name        VARCHAR,
        topic_prob        DOUBLE,
        source            VARCHAR,
        review_lemma      VARCHAR
    )
""")

print("Schema created successfully.")
for table in ["banks","topics","reviews"]:
    cols = con.execute(f"DESCRIBE {table}").fetchdf()
    print(f"  {table}: {len(cols)} columns")

Schema created successfully.
  banks: 4 columns
  topics: 4 columns
  reviews: 17 columns


---
## Step 5 — Insert Reference Tables

### Banks and topics before reviews

Foreign key constraints are enforced at insert time. The `banks` and `topics` reference tables must be populated before `reviews` — any review with an unresolvable `bank_id` or `topic_id` will throw a constraint violation.

Known store ratings come from the project brief (CBE=4.4, BOA=2.8, Dashen=4.0). Our scraped averages differ slightly because the store rating is an all-time cumulative average while we only scraped recent reviews.

In [35]:
BANK_META = {
    "BOA":    {"bank_id": 1, "app_id": "com.boa.boaMobileBanking",    "known_rating": 2.8},
    "CBE":    {"bank_id": 2, "app_id": "com.combanketh.mobilebanking","known_rating": 4.4},
    "Dashen": {"bank_id": 3, "app_id": "com.dashen.dashensuperapp",   "known_rating": 4.0},
}

for bank_name, meta in BANK_META.items():
    con.execute(
        "INSERT INTO banks VALUES (?, ?, ?, ?)",
        [meta["bank_id"], bank_name, meta["app_id"], meta["known_rating"]]
    )

# Always insert outlier topic first (-1 sentinel)
con.execute("INSERT INTO topics VALUES (-1, 'Uncategorized', 0, 'outlier')")

df_topics_meta = pd.read_csv(TOPICS_CSV)
for _, row in df_topics_meta.iterrows():
    tid    = int(row.get("Topic", row.get("topic_id", -99)))
    tname  = UNNAMED_TOPIC_MAP.get(str(row.get("topic_name","")), str(row.get("topic_name","")))
    tsize  = int(row.get("Count", 0))
    kwords = str(row.get("Name",""))
    con.execute("INSERT OR IGNORE INTO topics VALUES (?, ?, ?, ?)", [tid, tname, tsize, kwords])

print(f"Banks inserted   : {con.execute('SELECT COUNT(*) FROM banks').fetchone()[0]}")
print(f"Topics inserted  : {con.execute('SELECT COUNT(*) FROM topics').fetchone()[0]}")
print("\nBanks table:")
print(con.execute("SELECT * FROM banks").fetchdf().to_string(index=False))

Banks inserted   : 3
Topics inserted  : 17

Banks table:
 bank_id bank_name                       app_id  known_rating
       1       BOA     com.boa.boaMobileBanking           2.8
       2       CBE com.combanketh.mobilebanking           4.4
       3    Dashen    com.dashen.dashensuperapp           4.0


---
## Step 6 — Insert Reviews

### Why `INSERT INTO ... SELECT` from a registered DataFrame

DuckDB's `register()` exposes a pandas DataFrame as a virtual SQL table. A single SQL statement inserts all 2,282 rows in milliseconds. Row-by-row Python iteration would take minutes.

The `bank_id` is resolved via a JOIN inside the SQL: each review's `bank` string maps to the corresponding integer `bank_id` in the `banks` table.

In [36]:
con.register("df_reviews", df)

# Pre-validate bank names
unknown_banks = con.execute("""
    SELECT DISTINCT bank FROM df_reviews
    WHERE bank NOT IN (SELECT bank_name FROM banks)
""").fetchdf()
print(f"Unknown banks: {unknown_banks['bank'].tolist() if len(unknown_banks) > 0 else 'none — PASS'}")

# Pre-validate topic IDs and auto-insert missing ones
unknown_topics = con.execute("""
    SELECT DISTINCT topic_id FROM df_reviews
    WHERE topic_id NOT IN (SELECT topic_id FROM topics)
""").fetchdf()
if len(unknown_topics) > 0:
    for tid in unknown_topics["topic_id"].tolist():
        con.execute("INSERT OR IGNORE INTO topics VALUES (?, 'Uncategorized', 0, '')", [int(tid)])
    print(f"Auto-inserted {len(unknown_topics)} missing topic placeholders.")
else:
    print("Topic ID validation: PASS")

con.execute("""
    INSERT INTO reviews
    SELECT
        d.review_id,
        b.bank_id,
        CAST(d.topic_id AS INTEGER),
        d.review,
        CAST(d.rating AS INTEGER),
        CAST(d.date AS DATE),
        CAST(COALESCE(d.helpful_count, 0) AS INTEGER),
        d.language,
        d.sentiment_label,
        d.sentiment_score,
        CAST(d.low_confidence AS BOOLEAN),
        CAST(d.is_short_review AS BOOLEAN),
        CAST(d.developer_replied AS BOOLEAN),
        d.topic_name,
        d.topic_prob,
        d.source,
        d.review_lemma
    FROM df_reviews d
    JOIN banks b ON b.bank_name = d.bank
""")

row_count = con.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]
print(f"\nRows inserted    : {row_count:,}")
print(f"Expected         : {len(df):,}")
print(f"Status           : {'PASS' if row_count == len(df) else 'MISMATCH - check above errors'}") 

Unknown banks: none — PASS
Topic ID validation: PASS

Rows inserted    : 2,282
Expected         : 2,282
Status           : PASS


---
## Step 7 — Schema Verification

Four checks before writing any analytical queries:
1. Row counts per bank match expected
2. Foreign key integrity — no orphaned references
3. Rating constraint — all values between 1 and 5
4. NULL audit — expected ~539 rows without sentiment (non-English reviews)

In [37]:
print("=" * 55)
print("DATABASE VERIFICATION")
print("=" * 55)

counts = con.execute("""
    SELECT b.bank_name, COUNT(*) AS review_count
    FROM reviews r JOIN banks b USING (bank_id)
    GROUP BY b.bank_name ORDER BY b.bank_name
""").fetchdf()
print("\nRow counts by bank:")
print(counts.to_string(index=False))

orphan_banks  = con.execute("SELECT COUNT(*) FROM reviews WHERE bank_id  NOT IN (SELECT bank_id  FROM banks )").fetchone()[0]
orphan_topics = con.execute("SELECT COUNT(*) FROM reviews WHERE topic_id NOT IN (SELECT topic_id FROM topics)").fetchone()[0]
bad_ratings   = con.execute("SELECT COUNT(*) FROM reviews WHERE rating NOT BETWEEN 1 AND 5").fetchone()[0]
no_sentiment  = con.execute("SELECT COUNT(*) FROM reviews WHERE sentiment_label = ''").fetchone()[0]

print(f"\nFK bank_id       : {orphan_banks}  {'PASS' if orphan_banks  == 0 else 'FAIL'}")
print(f"FK topic_id      : {orphan_topics}  {'PASS' if orphan_topics == 0 else 'FAIL'}")
print(f"Bad ratings      : {bad_ratings}  {'PASS' if bad_ratings   == 0 else 'FAIL'}")
print(f"No-sentiment rows: {no_sentiment} (expected ~539 non-English/short reviews)")
print("\n" + "=" * 55)

DATABASE VERIFICATION

Row counts by bank:
bank_name  review_count
      BOA           790
      CBE           770
   Dashen           722

FK bank_id       : 0  PASS
FK topic_id      : 0  PASS
Bad ratings      : 0  PASS
No-sentiment rows: 548 (expected ~539 non-English/short reviews)



---
## Step 8 — Query 1: Sentiment Summary with Percentages

### Window function: `SUM() OVER (PARTITION BY)`

This computes each sentiment label as a percentage of its bank's total — without a subquery or self-join:

```sql
ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY bank_name), 1)
```

The `OVER (PARTITION BY bank_name)` window calculates the denominator separately per bank, inline with the main query. This is the most commonly used window function pattern in analytical SQL.

In [38]:
q1 = con.execute("""
    SELECT
        b.bank_name,
        r.sentiment_label,
        COUNT(*)                                                          AS review_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY b.bank_name), 1) AS pct_of_bank,
        ROUND(AVG(r.rating), 2)                                           AS avg_rating,
        ROUND(AVG(r.sentiment_score), 3)                                  AS avg_confidence
    FROM reviews r
    JOIN banks b USING (bank_id)
    WHERE r.sentiment_label != ''
    GROUP BY b.bank_name, r.sentiment_label
    ORDER BY b.bank_name, pct_of_bank DESC
""").fetchdf()

q1.to_csv(QUERY_DIR / "q1_sentiment_summary.csv", index=False)
print("Query 1 — Sentiment summary:")
print(q1.to_string(index=False))

Query 1 — Sentiment summary:
bank_name sentiment_label  review_count  pct_of_bank  avg_rating  avg_confidence
      BOA        negative           369         59.8        1.44           0.862
      BOA        positive           138         22.4        4.47           0.884
      BOA         neutral           110         17.8        3.23           0.683
      CBE        positive           248         46.3        4.68           0.892
      CBE        negative           174         32.5        2.20           0.788
      CBE         neutral           114         21.3        3.91           0.686
   Dashen        positive           369         63.5        4.86           0.916
   Dashen        negative           139         23.9        1.74           0.833
   Dashen         neutral            73         12.6        3.64           0.696


---
## Step 9 — Query 2: Monthly Trend with 3-Month Rolling Average

### Window function: `AVG() OVER (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`

A moving average computed entirely in SQL. The frame `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` looks at the current month plus the 2 months before it:

```sql
AVG(pct_negative) OVER (
    PARTITION BY bank_name   -- reset per bank
    ORDER BY month           -- define time ordering
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
)
```

This produces exactly what Task 4's time-series chart needs — pre-smoothed data that the visualization notebook just reads and plots.

In [39]:
q2 = con.execute("""
    WITH monthly_base AS (
        SELECT
            b.bank_name,
            STRFTIME(r.date, '%Y-%m')                    AS month,
            COUNT(*)                                      AS total_reviews,
            SUM(CASE WHEN r.sentiment_label = 'negative' THEN 1 ELSE 0 END) AS negative_count,
            SUM(CASE WHEN r.sentiment_label = 'positive' THEN 1 ELSE 0 END) AS positive_count,
            ROUND(AVG(r.rating), 2)                       AS avg_rating
        FROM reviews r
        JOIN banks b USING (bank_id)
        WHERE r.sentiment_label != ''
        GROUP BY b.bank_name, month
    ),
    monthly_pct AS (
        SELECT
            bank_name, month, total_reviews, avg_rating,
            ROUND(100.0 * negative_count / total_reviews, 1) AS pct_negative,
            ROUND(100.0 * positive_count / total_reviews, 1) AS pct_positive
        FROM monthly_base
    )
    SELECT
        bank_name, month, total_reviews, avg_rating,
        pct_negative, pct_positive,
        ROUND(AVG(pct_negative) OVER (
            PARTITION BY bank_name ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 1) AS pct_negative_3m_avg,
        ROUND(AVG(pct_positive) OVER (
            PARTITION BY bank_name ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 1) AS pct_positive_3m_avg
    FROM monthly_pct
    ORDER BY bank_name, month
""").fetchdf()

q2.to_csv(QUERY_DIR / "q2_monthly_sentiment_trend.csv", index=False)
print("Query 2 — Monthly trend (last 6 rows per bank):")
print(q2.groupby("bank_name").tail(3).to_string(index=False))

Query 2 — Monthly trend (last 6 rows per bank):
bank_name   month  total_reviews  avg_rating  pct_negative  pct_positive  pct_negative_3m_avg  pct_positive_3m_avg
      BOA 2026-01             21        2.19          66.7          19.0                 60.2                 21.8
      BOA 2026-02             12        3.00          66.7          25.0                 59.1                 25.1
      BOA 2026-03             30        3.77          23.3          43.3                 52.2                 29.1
      CBE 2026-01            109        3.74          40.4          42.2                 32.2                 46.8
      CBE 2026-02            147        3.66          34.0          48.3                 31.7                 50.3
      CBE 2026-03             68        3.51          35.3          41.2                 36.6                 43.9
   Dashen 2026-01             29        2.90          41.4          37.9                 33.8                 47.2
   Dashen 2026-02             17

---
## Step 10 — Query 3: Theme Pain Points and Drivers

### Window function: `RANK() OVER (PARTITION BY ... ORDER BY ...)`

`RANK()` assigns a rank within each bank's partition. Rank 1 by negative count = the most painful theme for that bank. Rank 1 by positive count = the biggest satisfaction driver.

This query directly answers the project requirement: *"Identify 2+ drivers and pain points per bank."*

In [40]:
q3 = con.execute("""
    WITH topic_sentiment AS (
        SELECT
            b.bank_name,
            r.topic_name,
            COUNT(*)                                              AS total,
            SUM(CASE WHEN r.sentiment_label = 'negative' THEN 1 ELSE 0 END) AS negative,
            SUM(CASE WHEN r.sentiment_label = 'positive' THEN 1 ELSE 0 END) AS positive,
            ROUND(AVG(r.rating), 2)                               AS avg_rating
        FROM reviews r
        JOIN banks b USING (bank_id)
        WHERE r.topic_id != -1 AND r.sentiment_label != ''
        GROUP BY b.bank_name, r.topic_name
    )
    SELECT
        bank_name, topic_name, total, avg_rating,
        ROUND(100.0 * negative / total, 1) AS pct_negative,
        ROUND(100.0 * positive / total, 1) AS pct_positive,
        RANK() OVER (PARTITION BY bank_name ORDER BY negative DESC) AS pain_rank,
        RANK() OVER (PARTITION BY bank_name ORDER BY positive DESC) AS driver_rank
    FROM topic_sentiment
    ORDER BY bank_name, pain_rank
""").fetchdf()

q3.to_csv(QUERY_DIR / "q3_theme_analysis.csv", index=False)
print("Top pain points (pain_rank <= 2):")
print(q3[q3["pain_rank"] <= 2][["bank_name","topic_name","total","pct_negative","pain_rank"]].to_string(index=False))
print("\nTop satisfaction drivers (driver_rank <= 2):")
print(q3[q3["driver_rank"] <= 2][["bank_name","topic_name","total","pct_positive","driver_rank"]].to_string(index=False))

Top pain points (pain_rank <= 2):
bank_name                 topic_name  total  pct_negative  pain_rank
      BOA General Banking Experience    153          90.8          1
      BOA     General App Complaints    107          86.9          2
      CBE General Banking Experience     75          82.7          1
      CBE     General App Complaints     77          58.4          2
   Dashen General Banking Experience     39          89.7          1
   Dashen     General App Complaints     91          37.4          2

Top satisfaction drivers (driver_rank <= 2):
bank_name                topic_name  total  pct_positive  driver_rank
      BOA     Speed and Convenience     57          64.9            1
      BOA Dashen Super App Feedback     44          81.8            2
      CBE     Speed and Convenience     79          83.5            2
      CBE Dashen Super App Feedback     82          89.0            1
   Dashen Positive General Feedback    126          88.9            1
   Dashen Dashen 

---
## Step 11 — Query 4: Weighted vs Unweighted Sentiment

### Why this query matters

Task 2 showed CBE's negative sentiment jumps from 33% (unweighted) to 83% (weighted by `helpful_count`). This query formalizes that finding in SQL and produces the `weight_gap` column — the number of percentage points by which the weighted result differs from the raw count.

A large positive `weight_gap` for negative sentiment means the reviews other users found most helpful are disproportionately negative — a stronger signal than raw counts suggest.

In [41]:
q4 = con.execute("""
    WITH weighted AS (
        SELECT
            b.bank_name,
            r.sentiment_label,
            COUNT(*)             AS review_count,
            SUM(r.helpful_count + 1) AS total_weight
        FROM reviews r
        JOIN banks b USING (bank_id)
        WHERE r.sentiment_label != ''
        GROUP BY b.bank_name, r.sentiment_label
    ),
    bank_totals AS (
        SELECT bank_name,
               SUM(review_count) AS total_reviews,
               SUM(total_weight) AS grand_weight
        FROM weighted GROUP BY bank_name
    )
    SELECT
        w.bank_name, w.sentiment_label, w.review_count,
        ROUND(100.0 * w.review_count / t.total_reviews, 1) AS pct_unweighted,
        ROUND(100.0 * w.total_weight  / t.grand_weight,  1) AS pct_weighted,
        ROUND((100.0 * w.total_weight / t.grand_weight) -
              (100.0 * w.review_count / t.total_reviews), 1) AS weight_gap
    FROM weighted w
    JOIN bank_totals t USING (bank_name)
    ORDER BY w.bank_name, w.sentiment_label
""").fetchdf()

q4.to_csv(QUERY_DIR / "q4_weighted_sentiment.csv", index=False)
print("Query 4 — Weighted vs unweighted sentiment:")
print(q4.to_string(index=False))
print("\nLargest weight gaps:")
print(q4.reindex(q4["weight_gap"].abs().sort_values(ascending=False).index)
    [["bank_name","sentiment_label","pct_unweighted","pct_weighted","weight_gap"]].head(4).to_string(index=False))

Query 4 — Weighted vs unweighted sentiment:
bank_name sentiment_label  review_count  pct_unweighted  pct_weighted  weight_gap
      BOA        negative           369            59.8          87.7        27.9
      BOA         neutral           110            17.8           6.2       -11.6
      BOA        positive           138            22.4           6.1       -16.3
      CBE        negative           174            32.5          83.2        50.7
      CBE         neutral           114            21.3           5.6       -15.7
      CBE        positive           248            46.3          11.3       -35.0
   Dashen        negative           139            23.9          35.3        11.4
   Dashen         neutral            73            12.6           1.7       -10.9
   Dashen        positive           369            63.5          63.0        -0.5

Largest weight gaps:
bank_name sentiment_label  pct_unweighted  pct_weighted  weight_gap
      CBE        negative            32.5     

---
## Step 12 — Query 5: Executive Scorecard

### `COUNT(*) FILTER (WHERE condition)` — clean conditional counting

`FILTER` is a concise SQL99 syntax that counts only rows matching a condition within an aggregate. It is equivalent to `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` but more readable.

This query produces the one-table executive summary that should lead the final report — every key metric per bank in a single row.

In [42]:
q5 = con.execute("""
    SELECT
        b.bank_name,
        b.known_rating,
        ROUND(AVG(r.rating), 2)                                            AS scraped_avg_rating,
        COUNT(*)                                                           AS total_reviews,
        COUNT(*) FILTER (WHERE r.sentiment_label = 'positive')             AS positive_count,
        COUNT(*) FILTER (WHERE r.sentiment_label = 'negative')             AS negative_count,
        ROUND(100.0 * COUNT(*) FILTER (WHERE r.sentiment_label = 'positive') /
              NULLIF(COUNT(*) FILTER (WHERE r.sentiment_label != ''), 0), 1) AS pct_positive,
        ROUND(100.0 * COUNT(*) FILTER (WHERE r.sentiment_label = 'negative') /
              NULLIF(COUNT(*) FILTER (WHERE r.sentiment_label != ''), 0), 1) AS pct_negative,
        COUNT(*) FILTER (WHERE r.developer_replied = TRUE)                 AS dev_replies,
        ROUND(100.0 * COUNT(*) FILTER (WHERE r.developer_replied = TRUE) /
              COUNT(*), 1)                                                 AS dev_reply_pct,
        ROUND(AVG(r.helpful_count), 1)                                     AS avg_helpful_votes,
        MIN(r.date)                                                        AS earliest_review,
        MAX(r.date)                                                        AS latest_review
    FROM reviews r
    JOIN banks b USING (bank_id)
    GROUP BY b.bank_name, b.known_rating
    ORDER BY pct_positive DESC
""").fetchdf()

q5.to_csv(QUERY_DIR / "q5_bank_scorecard.csv", index=False)
print("Query 5 — Executive scorecard:")
for col in q5.columns:
    print(f"  {col:<25}: {q5[col].tolist()}")

Query 5 — Executive scorecard:
  bank_name                : ['Dashen', 'CBE', 'BOA']
  known_rating             : [4.0, 4.4, 2.8]
  scraped_avg_rating       : [4.03, 3.89, 2.71]
  total_reviews            : [722, 770, 790]
  positive_count           : [369, 248, 138]
  negative_count           : [139, 174, 369]
  pct_positive             : [63.5, 46.3, 22.4]
  pct_negative             : [23.9, 32.5, 59.8]
  dev_replies              : [2, 0, 0]
  dev_reply_pct            : [0.3, 0.0, 0.0]
  avg_helpful_votes        : [16.0, 14.1, 8.1]
  earliest_review          : [Timestamp('2025-01-11 00:00:00'), Timestamp('2025-08-30 00:00:00'), Timestamp('2024-04-29 00:00:00')]
  latest_review            : [Timestamp('2026-03-17 00:00:00'), Timestamp('2026-03-21 00:00:00'), Timestamp('2026-03-20 00:00:00')]


---
## Step 13 — Query 6: Investigation Window vs Baseline

### Quantifying the temporal anomalies from Task 1

This query computes sentiment for each bank's identified investigation window and compares it to the overall baseline. The `sentiment_delta` column answers: was this period better or worse than average, and by how much?

A negative delta means the window was less negative than baseline (good). A positive delta means it was more negative (event-driven problem).

In [43]:
q6 = con.execute("""
    WITH baseline AS (
        SELECT
            b.bank_name,
            ROUND(100.0 * COUNT(*) FILTER (WHERE r.sentiment_label = 'negative') /
                  NULLIF(COUNT(*) FILTER (WHERE r.sentiment_label != ''), 0), 1) AS baseline_pct_neg
        FROM reviews r JOIN banks b USING (bank_id)
        GROUP BY b.bank_name
    ),
    windows AS (
        SELECT
            b.bank_name,
            CASE
                WHEN b.bank_name = 'BOA'    THEN 'May 2024 spike'
                WHEN b.bank_name = 'CBE'    THEN 'Aug 2025 inflection'
                WHEN b.bank_name = 'Dashen' THEN 'Jan-Apr 2025 launch'
            END AS window_label,
            COUNT(*) AS window_reviews,
            ROUND(AVG(r.rating), 2) AS window_avg_rating,
            ROUND(100.0 * COUNT(*) FILTER (WHERE r.sentiment_label = 'negative') /
                  NULLIF(COUNT(*) FILTER (WHERE r.sentiment_label != ''), 0), 1) AS window_pct_neg,
            ROUND(100.0 * COUNT(*) FILTER (WHERE r.sentiment_label = 'positive') /
                  NULLIF(COUNT(*) FILTER (WHERE r.sentiment_label != ''), 0), 1) AS window_pct_pos
        FROM reviews r JOIN banks b USING (bank_id)
        WHERE (
            (b.bank_name = 'BOA'    AND r.date BETWEEN '2024-04-01' AND '2024-06-30') OR
            (b.bank_name = 'CBE'    AND r.date BETWEEN '2025-08-01' AND '2026-03-31') OR
            (b.bank_name = 'Dashen' AND r.date BETWEEN '2025-01-01' AND '2025-05-31')
        )
        GROUP BY b.bank_name
    )
    SELECT
        w.bank_name, w.window_label, w.window_reviews,
        w.window_avg_rating, w.window_pct_neg, w.window_pct_pos,
        bl.baseline_pct_neg,
        ROUND(w.window_pct_neg - bl.baseline_pct_neg, 1) AS sentiment_delta
    FROM windows w JOIN baseline bl USING (bank_name)
    ORDER BY w.bank_name
""").fetchdf()

q6.to_csv(QUERY_DIR / "q6_investigation_windows.csv", index=False)
print("Query 6 — Investigation windows vs baseline:")
print(q6.to_string(index=False))
print("\nInterpretation:")
for _, row in q6.iterrows():
    direction = "MORE negative" if row["sentiment_delta"] > 0 else "LESS negative"
    print(f"  {row['bank_name']} ({row['window_label']}): "
          f"{abs(row['sentiment_delta']):.1f}pp {direction} than overall baseline")

Query 6 — Investigation windows vs baseline:
bank_name        window_label  window_reviews  window_avg_rating  window_pct_neg  window_pct_pos  baseline_pct_neg  sentiment_delta
      BOA      May 2024 spike             219               2.30            68.4            18.4              59.8              8.6
      CBE Aug 2025 inflection             770               3.89            32.5            46.3              32.5              0.0
   Dashen Jan-Apr 2025 launch             351               4.42            13.0            76.6              23.9            -10.9

Interpretation:
  BOA (May 2024 spike): 8.6pp MORE negative than overall baseline
  CBE (Aug 2025 inflection): 0.0pp LESS negative than overall baseline
  Dashen (Jan-Apr 2025 launch): 10.9pp LESS negative than overall baseline


---
## Step 14 — Export SQL Dump

### What the dump contains

```
1. CREATE TABLE statements  (schema documentation)
2. INSERT INTO banks        (3 rows)
3. INSERT INTO topics       (all topics)
4. INSERT INTO reviews      (500-row sample — full data in .duckdb)
```

The sample size keeps the file under ~500 KB for GitHub. The `.duckdb` binary file is the full dataset.

In [44]:
def esc(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "NULL"
    return "'" + str(val).replace("'", "''").replace("\n", " ")[:300] + "'"

with open(SQL_DUMP_PATH, "w", encoding="utf-8") as f:
    f.write(f"-- B5W2 Bank Reviews — SQL Dump\n")
    f.write(f"-- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"-- DuckDB v{duckdb.__version__}\n\n")

    f.write("CREATE TABLE banks (bank_id INTEGER PRIMARY KEY, bank_name VARCHAR NOT NULL, app_id VARCHAR NOT NULL, known_rating DOUBLE);\n\n")
    f.write("CREATE TABLE topics (topic_id INTEGER PRIMARY KEY, topic_name VARCHAR NOT NULL, topic_size INTEGER, keywords VARCHAR);\n\n")
    f.write("CREATE TABLE reviews (review_id VARCHAR PRIMARY KEY, bank_id INTEGER REFERENCES banks(bank_id), topic_id INTEGER REFERENCES topics(topic_id), review VARCHAR, rating INTEGER, date DATE, helpful_count INTEGER, language VARCHAR, sentiment_label VARCHAR, sentiment_score DOUBLE, low_confidence BOOLEAN, is_short_review BOOLEAN, developer_replied BOOLEAN, topic_name VARCHAR, topic_prob DOUBLE, source VARCHAR, review_lemma VARCHAR);\n\n")

    for _, row in con.execute("SELECT * FROM banks").fetchdf().iterrows():
        f.write(f"INSERT INTO banks VALUES ({row['bank_id']}, {esc(row['bank_name'])}, {esc(row['app_id'])}, {row['known_rating']});\n")

    f.write("\n")
    for _, row in con.execute("SELECT * FROM topics").fetchdf().iterrows():
        f.write(f"INSERT INTO topics VALUES ({int(row['topic_id'])}, {esc(row['topic_name'])}, {int(row['topic_size'] or 0)}, {esc(row['keywords'])});\n")

    f.write("\n-- Reviews sample (500 rows — full data in bank_reviews.duckdb)\n")
    sample = con.execute("SELECT * FROM reviews ORDER BY date DESC LIMIT 500").fetchdf()
    for _, row in sample.iterrows():
        f.write(f"INSERT INTO reviews VALUES ({esc(row['review_id'])}, {int(row['bank_id'])}, {int(row['topic_id'])}, {esc(row['review'])}, {int(row['rating'])}, {esc(str(row['date'])[:10])}, {int(row['helpful_count'] or 0)}, {esc(row['language'])}, {esc(row['sentiment_label'])}, {'NULL' if pd.isna(row['sentiment_score']) else row['sentiment_score']}, {str(bool(row['low_confidence'])).upper()}, {str(bool(row['is_short_review'])).upper()}, {str(bool(row['developer_replied'])).upper()}, {esc(row['topic_name'])}, {'NULL' if pd.isna(row['topic_prob']) else row['topic_prob']}, {esc(row['source'])}, {esc(str(row['review_lemma'])[:200])});\n")

dump_size = SQL_DUMP_PATH.stat().st_size / 1024
print(f"SQL dump written : {SQL_DUMP_PATH}")
print(f"Dump size        : {dump_size:.1f} KB")

SQL dump written : /kaggle/working/b5w2_outputs/bank_reviews_dump.sql
Dump size        : 133.7 KB


---
## Step 15 — Final Verification

In [45]:
con.close()
con = duckdb.connect(str(DB_PATH), read_only=True)

print("=" * 55)
print("TASK 3 — FINAL VERIFICATION")
print("=" * 55)

total  = con.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]
fk_b   = con.execute("SELECT COUNT(*) FROM reviews WHERE bank_id  NOT IN (SELECT bank_id  FROM banks )").fetchone()[0]
fk_t   = con.execute("SELECT COUNT(*) FROM reviews WHERE topic_id NOT IN (SELECT topic_id FROM topics)").fetchone()[0]
bad_r  = con.execute("SELECT COUNT(*) FROM reviews WHERE rating NOT BETWEEN 1 AND 5").fetchone()[0]

print(f"\nRows in database : {total:,}  {'PASS' if total == 2282 else 'MISMATCH'}")
print(f"FK bank_id       : {fk_b}  {'PASS' if fk_b == 0 else 'FAIL'}")
print(f"FK topic_id      : {fk_t}  {'PASS' if fk_t == 0 else 'FAIL'}")
print(f"Bad ratings      : {bad_r}  {'PASS' if bad_r == 0 else 'FAIL'}")

print("\nQuery result files:")
for f in sorted(QUERY_DIR.iterdir()):
    rows = len(pd.read_csv(f))
    print(f"  {f.name:<40} {rows} rows")

print(f"\nDatabase file    : {DB_PATH.stat().st_size/1024:.1f} KB")
print(f"SQL dump file    : {SQL_DUMP_PATH.stat().st_size/1024:.1f} KB")

print("\n" + "=" * 55)
print("Files ready for Task 4:")
for p in [DB_PATH, SQL_DUMP_PATH, *sorted(QUERY_DIR.iterdir())]:
    print(f"  {Path(p).name}")
print("=" * 55)
con.close()

TASK 3 — FINAL VERIFICATION

Rows in database : 2,282  PASS
FK bank_id       : 0  PASS
FK topic_id      : 0  PASS
Bad ratings      : 0  PASS

Query result files:
  q1_sentiment_summary.csv                 9 rows
  q2_monthly_sentiment_trend.csv           46 rows
  q3_theme_analysis.csv                    46 rows
  q4_weighted_sentiment.csv                9 rows
  q5_bank_scorecard.csv                    3 rows
  q6_investigation_windows.csv             3 rows

Database file    : 2316.0 KB
SQL dump file    : 133.7 KB

Files ready for Task 4:
  bank_reviews.duckdb
  bank_reviews_dump.sql
  q1_sentiment_summary.csv
  q2_monthly_sentiment_trend.csv
  q3_theme_analysis.csv
  q4_weighted_sentiment.csv
  q5_bank_scorecard.csv
  q6_investigation_windows.csv


---
## Summary

### Files created

| File | Purpose |
|---|---|
| `bank_reviews.duckdb` | Full database — submit with GitHub |
| `bank_reviews_dump.sql` | Schema + sample data — commit to repo |
| `q1_sentiment_summary.csv` | Per-bank sentiment % with avg confidence |
| `q2_monthly_sentiment_trend.csv` | Monthly trend + 3-month rolling average |
| `q3_theme_analysis.csv` | Pain point and driver rankings per bank |
| `q4_weighted_sentiment.csv` | Weighted vs unweighted sentiment gap |
| `q5_bank_scorecard.csv` | Executive summary table per bank |
| `q6_investigation_windows.csv` | Investigation window vs baseline delta |

### SQL patterns demonstrated

| Pattern | Query |
|---|---|
| `SUM() OVER (PARTITION BY)` | Q1 — % of total per group |
| `AVG() OVER (ROWS BETWEEN)` | Q2 — 3-month rolling average |
| `RANK() OVER (ORDER BY)` | Q3 — pain/driver rankings |
| Conditional aggregation | Q4 — weighted sentiment |
| `COUNT() FILTER (WHERE)` | Q5 — conditional counts |
| CTE + baseline delta | Q6 — window vs average |

---
**Next:** Task 4 — Load query result CSVs and build the final visualizations and report.